# Gradient Descent: correctness checks, tables, and plots
_Auto-generated on 2025-09-25 16:44:04._

This notebook implements and evaluates three algorithms from `MAIN.md`:
1) Constant-step gradient descent  
2) Gradient descent with Armijo backtracking  
3) Steepest Descent with exact line search (for quadratics)

We test on:
- a strongly convex quadratic $f(x)=\tfrac12 x^\top A x - b^\top x$ (with known minimizer),
- the 2D Rosenbrock function.

Outputs:
- Table of (step value $\lambda$, achieved accuracy $|f_k-f^*|$, iterations) for constant-step GD on the quadratic.
- Convergence plots of objective error and gradient norm per iteration for all three methods.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Callable, Dict, List
import pandas as pd

%matplotlib inline

@dataclass
class Hist:
    xs: List[np.ndarray]
    fs: List[float]
    grads: List[np.ndarray]
    lams: List[float]

def norm2(x: np.ndarray) -> float:
    return float(np.linalg.norm(x))

def run_fixed_step_gd(f: Callable[[np.ndarray], float],
                      grad: Callable[[np.ndarray], np.ndarray],
                      x0: np.ndarray,
                      lam: float,
                      maxit: int = 10_000,
                      tol: float = 1e-8) -> Hist:
    xs = [x0.copy()]
    fs = [f(x0)]
    grads = [grad(x0)]
    lams = []
    x = x0.copy()
    for k in range(maxit):
        g = grad(x)
        step = lam * g
        x_new = x - step
        xs.append(x_new.copy())
        fs.append(f(x_new))
        grads.append(grad(x_new))
        lams.append(lam)
        if norm2(x_new - x) <= tol or abs(fs[-1] - fs[-2]) <= tol or norm2(g) <= tol:
            break
        x = x_new
    return Hist(xs, fs, grads, lams)

def run_backtracking_armijo(f: Callable[[np.ndarray], float],
                            grad: Callable[[np.ndarray], np.ndarray],
                            x0: np.ndarray,
                            eps: float = 1e-4,
                            delta: float = 0.5,
                            t0: float = 1.0,
                            maxit: int = 10_000,
                            tol: float = 1e-8) -> Hist:
    xs = [x0.copy()]
    fs = [f(x0)]
    grads = [grad(x0)]
    lams = []
    x = x0.copy()
    for k in range(maxit):
        g = grad(x)
        fx = f(x)
        t = t0
        # Armijo: f(x - t g) <= f(x) - eps * t * ||g||^2
        while f(x - t * g) > fx - eps * t * (norm2(g) ** 2):
            t *= delta
            if t < 1e-16:
                break
        x_new = x - t * g
        xs.append(x_new.copy())
        fs.append(f(x_new))
        grads.append(grad(x_new))
        lams.append(t)
        if norm2(x_new - x) <= tol or abs(fs[-1] - fs[-2]) <= tol or norm2(g) <= tol:
            break
        x = x_new
    return Hist(xs, fs, grads, lams)

def run_steepest_descent_exact_quadratic(A: np.ndarray,
                                         b: np.ndarray,
                                         x0: np.ndarray,
                                         maxit: int = 10_000,
                                         tol: float = 1e-8) -> Hist:
    # f(x) = 0.5 x^T A x - b^T x ; grad = A x - b
    def f(x): return 0.5 * float(x.T @ A @ x) - float(b.T @ x)
    def g(x): return A @ x - b
    
    xs = [x0.copy()]
    fs = [f(x0)]
    grads = [g(x0)]
    lams = []
    x = x0.copy()
    for k in range(maxit):
        grad_k = g(x)
        if norm2(grad_k) <= tol:
            break
        # Exact step: alpha = (g^T g) / (g^T A g)
        alpha = float(grad_k.T @ grad_k) / float(grad_k.T @ A @ grad_k)
        x_new = x - alpha * grad_k
        xs.append(x_new.copy())
        fs.append(f(x_new))
        grads.append(g(x_new))
        lams.append(alpha)
        if norm2(x_new - x) <= tol or abs(fs[-1] - fs[-2]) <= tol:
            break
        x = x_new
    return Hist(xs, fs, grads, lams)


In [ ]:

# Strongly convex quadratic
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])
b = np.array([1.0, 1.0])

Ainv = np.linalg.inv(A)
xstar_quad = Ainv @ b

def f_quad(x: np.ndarray) -> float:
    return 0.5 * float(x.T @ A @ x) - float(b.T @ x)

def g_quad(x: np.ndarray) -> np.ndarray:
    return A @ x - b

fstar_quad = f_quad(xstar_quad)

# Rosenbrock (2D)
def f_rosen(x: np.ndarray) -> float:
    u, v = x[0], x[1]
    return (1 - u) ** 2 + 100.0 * (v - u ** 2) ** 2

def g_rosen(x: np.ndarray) -> np.ndarray:
    u, v = x[0], x[1]
    dfdu = -2 * (1 - u) - 400 * u * (v - u ** 2)
    dfdv = 200 * (v - u ** 2)
    return np.array([dfdu, dfdv])

xstar_rosen = np.array([1.0, 1.0])
fstar_rosen = 0.0


In [ ]:

# Starting points
x0_quad = np.array([2.5, -1.0])
x0_rosen = np.array([-1.2, 1.0])

# Fixed-step lambdas (theory: 0 < lambda < 2/L, L=max eigenvalue of A)
L_quad = max(np.linalg.eigvals(A)).real
lambda_grid = np.linspace(0.05, 1.9 / L_quad, 10)

rows = []
fixed_histories = {}
for lam in lambda_grid:
    hist = run_fixed_step_gd(f_quad, g_quad, x0_quad, lam=lam, maxit=10000, tol=1e-10)
    fixed_histories[lam] = hist
    final_f = hist.fs[-1]
    acc = abs(final_f - fstar_quad)
    rows.append({
        "Step value λ": lam,
        "Achieved accuracy |f_k - f*|": acc,
        "Number of iterations": len(hist.fs) - 1
    })

df_fixed = pd.DataFrame(rows).sort_values("Step value λ").reset_index(drop=True)
df_fixed


In [ ]:

def plot_error_vs_iterations(hist: Hist, fstar: float, title: str):
    errs = [abs(f - fstar) for f in hist.fs]
    plt.figure()
    plt.semilogy(range(len(errs)), errs)
    plt.xlabel("Iteration k")
    plt.ylabel("|f(x_k) - f*|")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_gradnorm_vs_iterations(hist: Hist, title: str):
    gnorms = [norm2(g) for g in hist.grads]
    plt.figure()
    plt.semilogy(range(len(gnorms)), gnorms)
    plt.xlabel("Iteration k")
    plt.ylabel("||∇f(x_k)||")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Backtracking on quadratic and Rosenbrock
bt_quad = run_backtracking_armijo(f_quad, g_quad, x0_quad, eps=1e-4, delta=0.5, t0=1.0, maxit=10000, tol=1e-10)
bt_rosen = run_backtracking_armijo(f_rosen, g_rosen, x0_rosen, eps=1e-4, delta=0.5, t0=1.0, maxit=100000, tol=1e-10)

# Steepest Descent with exact line search (quadratic)
sd_quad = run_steepest_descent_exact_quadratic(A, b, x0_quad, maxit=10000, tol=1e-12)

# Error plots (quadratic)
plot_error_vs_iterations(bt_quad, fstar_quad, "Quadratic: Backtracking GD objective error")
lam_small = list(fixed_histories.keys())[1]
lam_large = list(fixed_histories.keys())[-1]
plot_error_vs_iterations(fixed_histories[lam_small], fstar_quad, f"Quadratic: Fixed-step GD error (λ={lam_small:.4f})")
plot_error_vs_iterations(fixed_histories[lam_large], fstar_quad, f"Quadratic: Fixed-step GD error (λ={lam_large:.4f})")
plot_error_vs_iterations(sd_quad, fstar_quad, "Quadratic: Steepest Descent (exact line search) objective error")

# Gradient norm plots (quadratic)
plot_gradnorm_vs_iterations(bt_quad, "Quadratic: Backtracking GD gradient norm")
plot_gradnorm_vs_iterations(fixed_histories[lam_large], f"Quadratic: Fixed-step GD gradient norm (λ={lam_large:.4f})")
plot_gradnorm_vs_iterations(sd_quad, "Quadratic: Steepest Descent gradient norm")

# Rosenbrock plots
plot_error_vs_iterations(bt_rosen, fstar_rosen, "Rosenbrock: Backtracking GD objective error")
plot_gradnorm_vs_iterations(bt_rosen, "Rosenbrock: Backtracking GD gradient norm")


In [ ]:

# Save the table for convenience
import os
os.makedirs("/mnt/data", exist_ok=True)
csv_path = "/mnt/data/constant_step_quadratic_summary.csv"
df_fixed.to_csv(csv_path, index=False)
print("Saved table to:", csv_path)
